# Explore MCF skyddsrum API

In [ ]:
import httpx

response = httpx.get(
    "https://inspire.mcf.se/skyddsrum/ogc/features/v1",
    params = {"limit": 1}
)
response.json()


In [ ]:
response.headers.keys()

In [ ]:
response.status_code

## Explore data for coordinate areas

In [ ]:
re = httpx.get(
    "https://inspire.mcf.se/geoserver/skyddsrum/ogc/features/v1/collections/US_civilProtectionSite",
    params = {"f": "json"}
)
re.json()
data = re.json()

In [ ]:
type(data)

In [ ]:
data.keys()

In [ ]:
data["extent"]["spatial"]["bbox"]

### Claculate lon and lat:

format in API "[9.74245055, 55.17669713, 24.85581961, 67.9515059]"
- min lon, min lat, max lon, max lat
- center_lon = (min_lon + max_lon) / 2
- center_lat = (min_lat + max_lat) / 2
- center_point = (center_log, center_lat)

### Get lon and lat:
- min_lon = center_lon - d (distance)
- max_lon = center_lon + d
- min_lat = center_lat - d
- max_lat = center_lat + d

In [ ]:
# calculate
import math

def calculate_bbox(lon: float, lat: float, d_meter = 5000):
    d_lat = d_meter/111320
    d_lon = d_meter/(111320 * math.cos(math.radians(lat)))
    return [lon - d_lon, lat - d_lat, lon + d_lon, lat + d_lat]


In [ ]:
lon_c=18.0123
lat_c=59.3011
bbox = calculate_bbox(lon_c, lat_c)

In [ ]:
bbox

## Deeper exploration in data

In [ ]:
bbox_get = httpx.get(
        "https://inspire.mcf.se/geoserver/skyddsrum/ogc/features/v1/collections/US_civilProtectionSite/items",
        params = {
        "f": "json",
        "bbox": ",".join(map(str, bbox)),
        "limit": 100
        }
    )
bbox_get.json()
bbox_data = bbox_get.json()

In [ ]:
bbox_data.keys()

In [ ]:
bbox_data["features"]

## Function for closest skyddsrum

In [ ]:
# get skyddsrum in a box [min_lon, min_lat, max_lon, max_lat]

def get_data(bbox):
    bbox_get = httpx.get(
        "https://inspire.mcf.se/geoserver/skyddsrum/ogc/features/v1/collections/US_civilProtectionSite/items", # use /items to get the data
        params = {
        "f": "json",
        "bbox": ",".join(map(str, bbox)), # [a,b,c,d] -> "a,b,c,d" (to give a str til the API)
        }
    )
    data = bbox_get.json()

    # only include skyddsrum 
    data["features"] = [
        f for f in data.get("features", [])
        if f["properties"].get("statustype") == "Skyddsrum"
    ]
    return data

In [ ]:
# haversine: calculate the great-circle distance between two points on a sphere from their latitudes and longitudes 
# with regard to the globe beeing slimmer in towards the poles

# help from LLM to understand haversine and how to calculate it
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dp/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dl/2)**2
    return 2 * R * math.asin(math.sqrt(a))

In [ ]:
def closest_skyddsrum(lat_c, lon_c):

    bbox = calculate_bbox(lon=lon_c, lat=lat_c)
    data = get_data(bbox)

    # get features from API, empty list if empty
    features = data.get("features", [])

    
    result = []
    # read lon and lat for skyddsrum in bbox area
    for f in features:
        coords = f["geometry"]["coordinates"]
        lon_s, lat_s = coords[0][0], coords[0][1]

        # calculate distance to skyddsrum from input coordinates
        distance = haversine(lat_c, lon_c, lat_s, lon_s)
        result.append({
            "name": f["properties"].get("name", "unknown"),
            "lon": lon_s,
            "lat": lat_s,
            "distance_m": round(distance)
        })

    # return closest skyddsrum from input coordinates
    result.sort(key=lambda x: x["distance_m"])
    closest = result[:1]
    closest_lat = closest[0]["lat"]
    closest_lon = closest[0]["lon"]

    return closest_lat, closest_lon
    

In [ ]:
closest_skyddsrum(lon_c=lon_c, lat_c=lat_c)

## Flow of the tools
user address input (sweden_address_input()) -> get lon & lat -> closest_skyddsrum() -> get skyddsrum lon & lat -> 
function sweden API (sweden_skyddsrum_address()) -> get address for skyddsrum